# Tokenizer 훈련 및 Vocabulary 준비
- 한국어 위키백과 코퍼스 사용

## 1. 한국어 위키백과 Dataset 준비

한국어 위키백과 최신 dump 파일을 `CSV` 파일 형식으로 내려받는 법 
- github.com/paul-hyun/web-crawler를 cloning하던가 다운로드 받는다.
- Local PC의 web-crawler folder로 이동
- 사용할 conda 환경 `trans` activate시킨 다음,
- 설치 패키지: `tqdm`,`pandas`, `pandas`, `bs4`, `wget`, `pymongo`
- 터미널에서 $ python kowiki.py 실행
- kowiki 폴더아래 kowiki_yyyymmdd.csv 형태의 파일이 생성됨
- [NOTE] [위키백과: 데이터베이스 다운로드 ⟹ 한국어 위키백과 dump 파일의 종류](https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B2%A0%EC%9D%B4%EC%8A%A4_%EB%8B%A4%EC%9A%B4%EB%A1%9C%EB%93%9C)

#### kowiki.txt 로 변환
그러나, 실습 시간에 위의 과정을 실행하기에는 시간에 부족하고, 최근 파일도 달라지므로 Github의 NLP2024_1 사이트에서 `data/kowiki/kowiki.txt` 를 내려받아 사용하기로 한다. [다운로드 Link]()

In [1]:
import pandas as pd
import sentencepiece as spm
import torch
from torch import nn
import sys
import csv
import os
# sys.maxsize = 9223372036854775807 ==> C의 long int 보다 커서 error 남
csv.field_size_limit(170000)
in_file = "./kowiki/kowiki_20250506.csv"
out_file = "./kowiki/kowiki.txt"

if not os.path.exists(out_file):
    SEPARATOR = u"\u241D"  # Symbol For Group Separator ␝
    df = pd.read_csv(in_file, sep=SEPARATOR, engine="python")
    print(df.head())
    with open(out_file, "w", encoding='UTF8') as f:
        for index, row in df.iterrows():
            f.write(row["text"]) # title 과 text가 중복 되므로 text만 저장
            f.write("\n\n\n\n")  # 구분
    print(f"Creation of {out_file} is done.")
else:
    print(f"{out_file} already exists.")

./kowiki/kowiki.txt already exists.


## 2. Vocabulary 만들기

- `vocab_size` : ETRI KorBERT는 32,000개 SKT KoBERT는 8,000개를 사용.
- vocab_size가 커지면 성능이 좋아 지고 모델 파라미터 수가 증가한다.
- vocab 만들기가 성공적으로 끝나면, ./kowiki 폴더 아래에 `kowiki.model`과 `kowiki.vocab`이 생성된다.

In [2]:
%%time 
corpus = "./kowiki/kowiki.txt"
prefix = "./kowiki/kowiki"
vocab_size = 8000
spm.SentencePieceTrainer.train(
    f"--input={corpus} --model_prefix={prefix} --vocab_size={vocab_size + 7}" + 
    " --model_type=bpe" +
    " --max_sentence_length=999999" + # 문장 최대 길이
    " --pad_id=0 --pad_piece=[PAD]" + # pad (0)
    " --unk_id=1 --unk_piece=[UNK]" + # unknown (1)
    " --bos_id=2 --bos_piece=[BOS]" + # begin of sequence (2)
    " --eos_id=3 --eos_piece=[EOS]" + # end of sequence (3)
    " --user_defined_symbols=[SEP],[CLS],[MASK]") # 사용자 정의 토큰

CPU times: total: 2min 42s
Wall time: 2min 23s


#### Vocab 테스트

In [3]:
vocab_file = "./kowiki/kowiki.model"
vocab = spm.SentencePieceProcessor()
vocab.load(vocab_file)

lines = [
    "자연어처리에 관심이 많지만 좀 어렵다고 느껴집니다.",
    "너무 발전 속도가 빨라서 따라가기 벅차요.",
    "어떻게 하면 자연어처리 테크닉에 능숙할 수 있을까요?"
    ]

# text를 tensor로 변환
inputs = []
for line in lines:
    pieces = vocab.encode_as_pieces(line)
    ids = vocab.encode_as_ids(line)
    inputs.append(torch.tensor(ids))
    print(line)
    print(pieces)
    print(ids)
    print()

자연어처리에 관심이 많지만 좀 어렵다고 느껴집니다.
['▁자연', '어', '처', '리에', '▁관심', '이', '▁많', '지만', '▁좀', '▁어', '렵', '다고', '▁느', '껴', '집', '니다', '.']
[1145, 3754, 3969, 559, 3638, 3717, 210, 99, 3642, 137, 4465, 658, 1218, 5054, 3999, 1356, 3719]

너무 발전 속도가 빨라서 따라가기 벅차요.
['▁너무', '▁발전', '▁속', '도가', '▁빨', '라', '서', '▁따라', '가', '기', '▁', '벅', '차', '요', '.']
[2796, 1046, 293, 893, 2903, 3753, 3732, 264, 3728, 3735, 3716, 5160, 3872, 3886, 3719]

어떻게 하면 자연어처리 테크닉에 능숙할 수 있을까요?
['▁어떻게', '▁하', '면', '▁자연', '어', '처', '리', '▁테', '크', '닉', '에', '▁능', '숙', '할', '▁수', '▁있을', '까', '요', '?']
[3372, 29, 3833, 1145, 3754, 3969, 3738, 519, 3863, 4452, 3720, 1049, 4279, 3882, 18, 1413, 3925, 3886, 4406]



In [4]:
len(vocab)

8007